In [49]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [50]:
data = open('names.txt', 'r').read().splitlines()

In [51]:
clean = sorted(list(set(''.join(data))))

In [52]:
char_to_int = {s:i+1 for i, s in enumerate(clean)}
int_to_char = {i:s for s, i in char_to_int.items()}
char_to_int['.'] = 26


In [53]:
#Collect n input from users
n = int(input("Enter the value of n for n-gram model: "))

# Build n-grams
ngrams = {}
for d in data:
    characters = ['.'] + list(d) + ['.']
    for i in range(len(characters) - n + 1):
        ngram = tuple(characters[i:i+n])
        ngrams[ngram] = ngrams.get(ngram, 0) + 1



Enter the value of n for n-gram model: 2


In [54]:
# Convert n-grams to counts tensor
counts = sorted(ngrams.items(), key=lambda ky: -ky[1])
N = torch.zeros((27,) * n, dtype=torch.int32)

# Fill counts tensor
for ngram, count in ngrams.items():
    indices = tuple(char_to_int[ch] for ch in ngram)
    N[indices] = count



In [55]:
x, y = [], []
for d in data[:1]:
    characters = ['.'] + list(d) + ['.']
    for i in range(len(characters) - n):
        ngram = tuple(characters[i:i+n])
        x.append(tuple(char_to_int[ch] for ch in ngram[1:]))  # Input sequence
        y.append(char_to_int[characters[i + n]])  # Target character
x = torch.tensor(x)
y = torch.tensor(y)

encoding = F.one_hot(x, num_classes=27).float()
w = torch.randn((27,) * n)

In [56]:
num_epochs = 10
learning_rate = 0.001
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.SGD([w], lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    # Forward pass
    predictions = encoding @ w
    
    # Compute the loss
    loss = loss_function(predictions, N.float()); loss.requires_grad = True
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
    
    log_probs = F.log_softmax(predictions, dim=-1)

    # Reshape the counts tensor to match the shape of log_probs
    reshaped_counts = N.view(-1, N.size(-1))

    # Compute NLL
    nll_loss = -torch.sum(log_probs * reshaped_counts.float()) / reshaped_counts.sum()

    print(f'NLL Loss: {nll_loss.item()}')



NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
NLL Loss: 15.161596298217773
Epoch [10/10], Loss: 338922.3750
NLL Loss: 15.161596298217773
